# 05 - Ablation, seed robustness, latency, evidence manifest
* Which feature group drives RCAK? (drop-one / only-one ablation, iso-budget metric)
* Is the gain stable across 5 different article-level splits **and** query samples (retrievers refit each time)?
* What does the controller cost in milliseconds per query?
* SHA-256 manifest of every output file (tamper-evident evidence trail for the thesis).

**Outputs:** `csv/ablation.csv`, `csv/seed_robustness.csv`, `csv/latency.csv`, `MANIFEST.csv`

In [1]:
# ---- Environment setup (works in Colab AND in a local venv / Jupyter) ----
import os, sys, pathlib, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "sentence-transformers", "python-docx", "psutil"], check=False)
    if not pathlib.Path("adaptive_rag").exists():
        from google.colab import files
        print("Upload rag_adaptive_project.zip (the project archive you downloaded)")
        up = files.upload(); z = [k for k in up if k.endswith(".zip")][0]
        zipfile.ZipFile(z).extractall("."); 
        root = [p for p in pathlib.Path(".").iterdir() if (p / "adaptive_rag").exists()][0]; os.chdir(root)
    try:
        from google.colab import drive; drive.mount("/content/drive")      # results persist on your Drive
        os.environ["RAG_RESULTS_DIR"] = "/content/drive/MyDrive/rag_adaptive_results"
    except Exception as e:
        os.environ["RAG_RESULTS_DIR"] = os.path.abspath("results")
    BACKEND = "st"          # real neural embeddings on Colab (GPU recommended: Runtime > Change runtime type > T4)
else:
    here = pathlib.Path.cwd()
    root = here.parent if here.name == "notebooks" else here
    sys.path.insert(0, str(root)); os.chdir(root)
    os.environ.setdefault("RAG_RESULTS_DIR", str(root / "results_notebooks"))
    BACKEND = "lsa"         # offline CPU baseline (TF-IDF+SVD). Set to "st" if sentence-transformers can download models.
sys.path.insert(0, os.getcwd())
CFG = dict(backend=BACKEND, st_model="sentence-transformers/all-MiniLM-L6-v2", n_queries=int(os.environ.get("RAG_NQ", 10000)), seed=0)
print("backend:", BACKEND, "| results ->", os.environ["RAG_RESULTS_DIR"])


backend: lsa | results -> /home/claude/rag_adaptive_project/results_notebooks


In [2]:
import pandas as pd
from adaptive_rag import experiment as E, plots as P
from adaptive_rag.utils import results_dir, write_manifest, log_event
B = E.prepare_cached(CFG)
ab = E.ablation(B, n_boot=300); ab.to_csv(results_dir("csv")/"ablation.csv", index=False); P.fig_ablation(ab); display(ab.round(3))

,variant,n_features,mean_k,answer_in_context,fixed_at_same_k,diff,ci95_lo,ci95_hi
0,all features,26,19.487,0.911,0.868,0.043,0.028,0.056
1,without dense_scores,18,19.682,0.913,0.870,0.043,0.028,0.055
2,without bm25_scores,21,23.638,0.919,0.896,0.022,0.013,0.032
3,without agreement,22,22.554,0.891,0.892,-0.001,-0.013,0.014
4,without query,17,19.582,0.913,0.869,0.044,0.028,0.057
5,only dense_scores,8,30.224,0.915,0.923,-0.008,-0.013,-0.002
6,only bm25_scores,5,22.576,0.892,0.892,-0.000,-0.011,0.013
7,only agreement,4,24.982,0.911,0.904,0.007,-0.003,0.019
8,only query,9,26.981,0.913,0.912,0.001,-0.008,0.007


In [3]:
lat = E.latency(B); lat.to_csv(results_dir("csv")/"latency.csv", index=False); P.fig_latency(lat); display(lat.round(3))

,component,mean_ms,p50_ms,p95_ms
0,"lsa retrieval (1 query, top-50)",71.966,70.120,85.249
1,"bm25 retrieval (1 query, top-50)",0.412,0.394,0.551
2,"RCAK controller (features + prediction, 1 query)",3.848,3.679,4.825


In [4]:
sd = E.seeds(CFG, seed_list=(0,1,2,3,4)); sd.to_csv(results_dir("csv")/"seed_robustness.csv", index=False); P.fig_seeds(sd)
display(sd.groupby("strategy")[["mean_k","answer_in_context","fixed_at_same_k","diff"]].agg(["mean","std"]).round(3))

mean_k        answer_in_context        fixed_at_same_k  \
                   mean    std              mean    std            mean   
strategy                                                                  
CGC (proposed)   13.065  0.227             0.854  0.003           0.806   
RCAK (proposed)  19.296  0.302             0.914  0.003           0.872   
adaptive_k_gap   21.448  0.149             0.889  0.006           0.889   

                         diff         
                   std   mean    std  
strategy                              
CGC (proposed)   0.003  0.048  0.004  
RCAK (proposed)  0.004  0.041  0.004  
adaptive_k_gap   0.003 -0.000  0.007

In [5]:
m = write_manifest(); log_event("notebook05_done"); display(pd.read_csv(m).head(20))

,file,bytes,sha256
0,csv/ablation.csv,1327,6ab714092f8dc6a3ec3ce291b41568937c740873625951...
1,csv/build_timing.csv,174,35e22bd707568fbcf018ba1a37a41720e2eb331370950c...
2,csv/dataset_stats.csv,264,0bc36092275fdbc84fd4162a441a7bf289700b8a107ed9...
3,csv/feature_matrix_sample.csv,62228,eabf115aac2b7bf71093cdb9bfa02468d888efbc48bb15...
4,csv/fig_ablation.csv,1327,6ab714092f8dc6a3ec3ce291b41568937c740873625951...
5,csv/fig_coverage_bm25.csv,1313,d56cc049af061ec218481c238bf85b60f092ac03667a3f...
6,csv/fig_coverage_hybrid_rrf.csv,1364,59aed5dc06bfac97f6ea61380796a912e14731207120b3...
7,csv/fig_coverage_lsa.csv,1206,073a99ce4e43984a23aa0ff21408acb4ed17dbe20803b8...
8,csv/fig_isobudget_answer_in_context_bm25.csv,7059,51c529556db0712c99c77ce27255d2f5aa04958567e4a2...
9,csv/fig_isobudget_answer_in_context_hybrid_rrf...,7311,b712f93db65e883277426931e8622ee4ef90d11b06fe52...
